# LP3 Machine Learning Practical 1 - Uber Fare Prediction

**Problem statement:** Predict the price of the Uber ride from a given pickup point to the agreed drop-off location. Tasks: 1. Pre-process the dataset. 2. Identify outliers. 3. Check the correlation. 4. Implement linear regression and random forest regression models. 5. Evaluate the models and compare their respective scores like R2, RMSE, etc. Dataset: https://www.kaggle.com/datasets/yasserh/uber-fares-dataset

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# 1. Load data
df = pd.read_csv('/content/uber.csv')
print("Dataset head:")
print(df.head())

In [ ]:
# 2. Preprocess data
# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
# Drop rows with missing values if any
df = df.dropna()

# Feature engineering: Convert datetime to useful features
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['hour'] = df['pickup_datetime'].dt.hour
df['dayofweek'] = df['pickup_datetime'].dt.dayofweek

In [ ]:
# Calculate haversine distance between pickup and dropoff
def haversine(lat1, lon1, lat2, lon2):
    # convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of earth in kilometers
    return c * r

df['distance_km'] = haversine(df['pickup_latitude'], df['pickup_longitude'],
                              df['dropoff_latitude'], df['dropoff_longitude'])

In [ ]:
# Remove entries with zero or near-zero distance or fare (likely errors)
df = df[(df['distance_km'] > 0.01) & (df['fare_amount'] > 0)]

print(f"\nData shape after cleaning: {df.shape}")

In [ ]:
# 3. Outlier detection - visualize fare and distance
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
sns.boxplot(x=df['fare_amount'])
plt.title("Fare Amount Distribution")

plt.subplot(1,2,2)
sns.boxplot(x=df['distance_km'])
plt.title("Distance Distribution (km)")

plt.show()

fare_99 = df['fare_amount'].quantile(0.99)
dist_99 = df['distance_km'].quantile(0.99)

df = df[(df['fare_amount'] <= fare_99) & (df['distance_km'] <= dist_99)]
print(f"\nData shape after removing outliers: {df.shape}")

In [ ]:
# 4. Correlation heatmap
plt.figure(figsize=(8,6))
sns.heatmap(df[['fare_amount', 'distance_km', 'hour', 'dayofweek']].corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation")
plt.show()

In [ ]:
# 5. Model preparation
features = ['distance_km', 'hour', 'dayofweek']
X = df[features]
y = df['fare_amount']

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Linear Regression model
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [ ]:
# 7. Random Forest Regression model
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [ ]:
# 8. Evaluation function
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} - RMSE: {rmse:.2f}, R2: {r2:.4f}")

evaluate(y_test, y_pred_lr, "Linear Regression")
evaluate(y_test, y_pred_rf, "Random Forest")

In [ ]:
# 9. Visualize predictions vs actual for Random Forest (best model expected)
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred_rf, alpha=0.3)
plt.xlabel("Actual Fare")
plt.ylabel("Predicted Fare")
plt.title("Random Forest Regression: Actual vs Predicted Fare")
plt.plot([0, y_test.max()], [0, y_test.max()], 'r--')  # diagonal line
plt.show()

## Theory and Explanation
The last cell contains the theory, steps, result interpretation and viva questions as comments.

In [ ]:
# =====================================================================
# THEORY AND EXPLANATION - Practical 1: Uber Fare Prediction
# =====================================================================
#
# 1. PROBLEM
# Predict fare_amount (a continuous number) from the trip details. Because the
# target is a continuous value, this is a REGRESSION problem (supervised learning).
#
# 2. KEY THEORY
# a) Data pre-processing
#    Raw data has missing values, impossible values and unusable formats. Here we
#    drop null rows, convert the text datetime into a real datetime and extract
#    hour and day of week, build a distance feature, and remove invalid records.
#
# b) Haversine distance
#    Great-circle distance between two latitude/longitude points on a sphere:
#        a = sin^2(dlat/2) + cos(lat1) * cos(lat2) * sin^2(dlon/2)
#        distance = 2 * R * arcsin(sqrt(a)),  R = 6371 km
#    Raw coordinates are not directly useful to the model; the distance travelled
#    is what drives the fare.
#
# c) Outliers
#    Values far away from the rest of the data (for example a fare above 200 or a
#    distance of 10000+ km, which is clearly a data error). They are identified
#    visually with box plots (points beyond the 1.5 * IQR whiskers). Here they are
#    removed by keeping only values at or below the 99th percentile.
#
# d) Correlation
#    Pearson correlation r lies between -1 and +1 and measures the strength of the
#    linear relationship between two variables. Near +1 or -1 means strong, near 0
#    means no linear relationship. The heatmap shows r for every pair of features.
#
# e) Linear Regression
#    Fits y = b0 + b1*x1 + b2*x2 + ... using Ordinary Least Squares, i.e. it picks
#    the coefficients that minimise the sum of squared errors.
#
# f) Random Forest Regression
#    An ensemble of many decision trees (n_estimators = 100). Each tree is trained
#    on a bootstrap sample of the data using random subsets of features (bagging).
#    The final prediction is the average of all the trees. It captures non-linear
#    relationships and overfits less than a single decision tree.
#
# g) Evaluation metrics
#    MSE  = mean( (y - y_pred)^2 )
#    RMSE = sqrt(MSE). Same unit as the fare (dollars). Lower is better.
#    R2   = 1 - SS_res / SS_tot. The fraction of the variance in fare explained by
#           the model. 1 is perfect, 0 is no better than always predicting the mean.
#
# 3. STEPS (code flow)
#    1. Load the CSV and check for nulls (only 1 missing dropoff latitude and
#       longitude), then drop those rows.
#    2. Convert pickup_datetime to datetime and extract hour and dayofweek.
#    3. Compute distance_km with the haversine function.
#    4. Remove rows with distance <= 0.01 km or fare <= 0 (bad records).
#       Rows left: 193,868.
#    5. Draw box plots to see outliers, then drop rows above the 99th percentile
#       of fare and distance. Rows left: 190,709.
#    6. Plot the correlation heatmap.
#    7. Features = distance_km, hour, dayofweek. Target = fare_amount.
#    8. Split 80/20 into train and test (random_state = 42 makes it repeatable).
#    9. Train Linear Regression and Random Forest, predict on the test set.
#   10. Compute RMSE and R2 for both and plot actual versus predicted fares.
#
# 4. RESULTS AND HOW TO READ THEM
#    Correlation with fare_amount: distance_km 0.88 (strong), hour -0.021 and
#    dayofweek 0.0064 (almost none). Distance is the main driver of the fare.
#
#    Linear Regression : RMSE 3.65, R2 0.7715
#    Random Forest     : RMSE 3.77, R2 0.7566
#
#    Both models explain about 76 to 77 percent of the variation in fare, with an
#    average error of roughly 3.7 dollars. Linear Regression is slightly better
#    here. This is reasonable: the fare is close to a straight-line function of
#    distance and only one feature carries real signal, so the extra flexibility of
#    the forest brings no benefit. The comment "best model expected" in the last
#    code cell was only an expectation; the numbers show Linear Regression wins.
#
#    In the actual versus predicted plot, points near the red diagonal are good
#    predictions. The vertical stripes at fixed actual fares (around 45 to 52) are
#    probably flat-rate trips that distance alone cannot explain.
#
#    Ways to improve: add location features (for example airport flags), the year
#    of the trip (the dataset spans several years, fares changed over time),
#    passenger_count, and tune the forest hyperparameters.
#
# 5. VIVA QUESTIONS
#    Q: Why remove outliers?
#    A: They distort the fit, especially for linear regression, which is sensitive
#       to extreme values. Some here are plain data errors.
#    Q: Why use RMSE and R2 and not accuracy?
#    A: Accuracy is for classification. For a continuous target we measure error
#       size (RMSE) and explained variance (R2).
#    Q: Does Random Forest need feature scaling?
#    A: No. Trees split on thresholds, so feature scale does not matter.
#    Q: What is bagging?
#    A: Training each model on a random bootstrap sample and averaging the results
#       to reduce variance.
#    Q: What does random_state do?
#    A: Fixes the random seed so the split and the forest give the same result
#       every run.
#    Q: What is overfitting?
#    A: The model learns noise in the training data, so train score is high but
#       test score is low.